In [0]:
airbnb_guests = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/guests.csv', header=True)
# display(airbnb_guests)
airbnb_hosts = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/hosts.csv', header=True)
# display(airbnb_hosts)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window 

result = airbnb_guests.join(airbnb_hosts, 
                            (airbnb_guests['nationality'] == airbnb_hosts['nationality']) & 
                            (airbnb_guests['gender'] == airbnb_hosts['gender']), 
                            'inner')

final_result = result.select('host_id', 'guest_id').distinct().orderBy('host_id', 'guest_id')

display(final_result)

host_id,guest_id
0,9
1,5
10,6
11,11
2,1
3,7
4,0
5,2
6,4
7,10


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

yelp_reviews = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/yelp_reviews.csv', header=True)
# yelp_reviews.show()

WindowSpec = Window.orderBy(desc(col('funny')))
yelp_reviews1 = yelp_reviews.withColumn('rnk', dense_rank().over(WindowSpec))
yelp_reviews2 = yelp_reviews1.filter(col('rnk') == 2)
yelp_reviews3 = yelp_reviews2.select('business_name', 'review_text')
display(yelp_reviews3)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


business_name,review_text
Flancer's Cafe,"This place has decent food, cute atmosphere, but the service is problematic. I was stuck in Mesa for training and had lunch there on Halloween. My pal"
Roka Akor,"I hate to admit it, but it had been a long while since my last visit to Roka Akor. I deserve a hand slap. But last week, I had the perfect excuse to p"


In [0]:
events = spark.read.csv('/Volumes/workspace/stratascratch/stratascratch/events.csv', header=True)
display(events, showtruncate=True)

user_id,occurred_at,event_type,event_name,location,device
6991,2014-06-09 18:26:54,engagement,home_page,United States,iphone 5
18851,2014-08-29 13:18:38,signup_flow,enter_info,Russia,asus chromebook
14998,2014-07-01 12:47:56,engagement,login,France,hp pavilion desktop
8186,2014-05-23 10:44:16,engagement,home_page,Italy,macbook pro
9626,2014-07-31 17:15:14,engagement,login,Russia,nexus 7
16460,2014-07-24 18:43:19,signup_flow,create_user,United States,samsung galaxy note
10101,2014-08-27 05:54:28,engagement,home_page,Singapore,dell inspiron notebook
2670,2014-05-10 10:03:34,engagement,like_message,United States,nexus 7
8708,2014-05-26 10:42:12,engagement,send_message,Australia,macbook pro
167,2014-07-30 19:39:13,engagement,view_inbox,United Arab Emirates,lenovo thinkpad


In [0]:
events = spark.read.format('csv').option('header', 'true').option('inferschema', 'true').option('mode', 'PERMISSIVE').load('/Volumes/workspace/stratascratch/stratascratch/events.csv')

#bronze_state = raw_data
# display(events, showtruncate=True) 

from pyspark.sql.functions import *
from pyspark.sql.window import Window
import re 

#silver_state = clean-up all null values and negative ids, and remove duplicates
events_silver = events.filter((col('user_id').isNotNull()) & (expr("try_cast(user_id as int) > 0"))).dropDuplicates()
# display(events_silver)

#gold_state
WindowSpec = Window.partitionBy('location')
events_gold1 = events_silver.filter(col('device').rlike('^samsung'))\
    .withColumn('number_of_users', count('user_id').over(WindowSpec))
events_gold2 = events_silver.filter((col('device').rlike('^iphone')) | (col('device').rlike('^macbook')))\
    .withColumn('number_of_users', count('user_id').over(WindowSpec))

display(events_gold2)


user_id,occurred_at,event_type,event_name,location,device,number_of_users
99999,2014-07-28 17:10:00,engagement,home_page,Argentina,macbook pro,14
16170,2014-08-25 13:32:34,engagement,home_page,Argentina,macbook pro,14
12103,2014-06-19 19:45:39,engagement,home_page,Argentina,macbook pro,14
12103,2014-06-25 11:10:04,engagement,like_message,Argentina,macbook pro,14
251,2014-08-02 10:47:41,engagement,login,Argentina,macbook air,14
16170,2014-08-19 11:07:59,engagement,login,Argentina,macbook pro,14
251,2014-08-06 15:24:42,engagement,login,Argentina,macbook air,14
16170,2014-08-23 18:53:20,engagement,like_message,Argentina,macbook pro,14
12103,2014-06-25 11:08:39,engagement,like_message,Argentina,macbook pro,14
12103,2014-06-25 11:07:03,engagement,login,Argentina,macbook pro,14


In [0]:
department_data = [
    (1, "IT"),
    (2, "Sales")
]

department_schema = ["id", "name"]

employee_data = [
    (1, "Joe", 85000, 1),
    (2, "Henry", 80000, 2),
    (3, "Sam", 60000, 2),
    (4, "Max", 90000, 1),
    (5, "Janet", 69000, 1),
    (6, "Randy", 85000, 1),
    (7, "Will", 70000, 1)
]

employee_schema = ["id", "name", "salary", "departmentId"]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

department = spark.createDataFrame(department_data, department_schema)
employee = spark.createDataFrame(employee_data, employee_schema)

WindowSpec1 = Window.partitionBy('departmentId').orderBy(desc('salary'))

pre_result = employee.join(department, employee['departmentId'] == department['id'], 'left').withColumn('rnk', dense_rank().over(WindowSpec1))
result = pre_result.filter(col('rnk')<=3).select(employee['name'].alias('Employee'), department['name'].alias('Department'), employee['salary'].alias('Salary')).orderBy('Department', desc('salary'), 'Employee')
display(result)

Employee,Department,Salary
Max,IT,90000
Joe,IT,85000
Randy,IT,85000
Will,IT,70000
Henry,Sales,80000
Sam,Sales,60000


In [0]:
person_data = [
    (1, "john@example.com"),
    (2, "bob@example.com"),
    (3, "john@example.com"),
    (4, "alice@example.com"),
    (5, "bob@example.com"),
    (6, "john@example.com"),
    (7, "charlie@example.com"),
    (8, "alice@example.com"),
    (9, "david@example.com"),
    (10, "bob@example.com"),
    (11, "charlie@example.com"),
    (12, "john@example.com"),
    (13, "eve@example.com"),
    (14, "alice@example.com"),
    (15, "david@example.com"),
    (16, "john@example.com"),
]

person_schema = ["id", "email"]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

person = spark.createDataFrame(person_data, person_schema)
# display(person)
WindowSpec = Window.partitionBy('email').orderBy('id')
person1 = person.withColumn('rnk', rank().over(WindowSpec))
person2 = person1.filter(col('rnk') == 1)
display(person2.select('id', 'email').orderBy('id'))


id,email
1,john@example.com
2,bob@example.com
4,alice@example.com
7,charlie@example.com
9,david@example.com
13,eve@example.com


In [0]:
from pyspark.sql.functions import *

us_high_growth_micro_small_cap_stocks = spark.read.option("multiLine", "true").json("/Volumes/workspace/stratascratch/usstockmarket/us_high_growth_micro_small_cap_stocks.json")

display(us_high_growth_micro_small_cap_stocks)

analyst_consensus,analyst_price_target,analyst_reviews,currency,exchange,growth_opportunity,investment_role,long_term_thesis,market_cap_category,notes,number_of_buy_signal,number_of_hold_signals,number_of_sell_signal,risk_level,sector,stock_name,theme,ticker
Buy,79.87,Buy consensus,USD,NASDAQ,Very High,High-growth space/defense core,"Neutron, launch services, satellite systems and defense contracts could materially expand Rocket Lab's addressable market.",Mid Cap,No longer a micro/small cap; it has grown substantially.,26,null,null,Very High,Aerospace & Defense,Rocket Lab Corporation,"Space launch, satellites, defense systems",RKLB
null,null,Strong institutional interest,null,NASDAQ,Very High,AI infrastructure growth,"AI data centers require rapidly increasing bandwidth and connectivity, supporting demand for high-speed connectivity solutions.",null,null,null,null,null,High,Semiconductors,Credo Technology Group Holding Ltd,AI data-center connectivity,CRDO
null,null,Needs continuous monitoring,null,NASDAQ,Very High,High-risk AI infrastructure satellite,Rising data-center bandwidth requirements can drive demand for optical components and networking products.,null,Highly volatile; position sizing matters.,null,null,null,Very High,Communications Equipment,Applied Optoelectronics Inc.,Optical networking and AI data centers,AAOI
null,null,Growth-focused coverage,null,NASDAQ,Very High,10-year asymmetric growth bet,Commercial autonomous trucking could create a large software/service market with potential fleet productivity and operating-cost benefits.,null,null,null,null,null,Very High,Autonomous Vehicles,Aurora Innovation Inc.,Autonomous trucking,AUR
Buy,84.2,Buy consensus,USD,NYSE,Very High,Speculative nuclear/AI-power moonshot,"Growing electricity demand, including from data centers, could create a major market opportunity for advanced nuclear generation.",null,Analyst counts vary by provider and date; values reflect a recent 2026 source.,15,9,1,Extreme,Nuclear Energy,Oklo Inc.,Advanced nuclear power for rising electricity demand,OKLO
null,null,Clinical-outcome dependent,null,NASDAQ,Very High,Biotech moonshot,Successful gene-editing therapies could address diseases with significant unmet medical need and create substantial commercial opportunities.,null,null,null,null,null,Extreme,Biotechnology,Intellia Therapeutics Inc.,CRISPR gene editing,NTLA
null,null,null,null,NYSE,High,Smaller space-infrastructure alternative,Increasing government and commercial spending on space infrastructure can support long-term growth.,null,null,null,null,null,Very High,Aerospace & Defense,Redwire Corporation,Space infrastructure and defense,RDW
null,null,null,null,NASDAQ,Very High,Tiny-cap robotics optionality,Autonomous last-mile delivery could become a large robotics market if unit economics and deployment scale improve.,null,Highly speculative; suitable only for a small satellite allocation.,null,null,null,Extreme,Robotics,Serve Robotics Inc.,Autonomous delivery robots,SERV
null,null,null,null,NASDAQ,High,AI application growth,"Voice and conversational AI adoption across automotive, restaurants and enterprise applications could support significant growth.",null,Valuation and competitive intensity should be monitored.,null,null,null,Very High,Artificial Intelligence,SoundHound AI Inc.,Conversational and voice AI,SOUN


In [0]:
movies_data = [
    (1, "Avengers"),
    (2, "Frozen 2"),
    (3, "Joker")
]

movies_schema = ["movie_id", "title"]

users_data = [
    (1, "Daniel"),
    (2, "Monica"),
    (3, "Maria"),
    (4, "James")
]

users_schema = ["user_id", "name"]

movie_rating_data = [
    (1, 1, 3, "2020-01-12"),
    (1, 2, 4, "2020-02-11"),
    (1, 3, 2, "2020-02-12"),
    (1, 4, 1, "2020-01-01"),
    (2, 1, 5, "2020-02-17"),
    (2, 2, 2, "2020-02-01"),
    (2, 3, 2, "2020-03-01"),
    (3, 1, 3, "2020-02-22"),
    (3, 2, 4, "2020-02-25")
]

movie_rating_schema = [
    "movie_id",
    "user_id",
    "rating",
    "created_at"
]

movies = spark.createDataFrame(movies_data, movies_schema)
users = spark.createDataFrame(users_data, users_schema)
movie_rating = spark.createDataFrame(movie_rating_data, movie_rating_schema)

from pyspark.sql.functions import *
from pyspark.sql.types import *

df_1 = movies.join(movie_rating, movies['movie_id'] == movie_rating['movie_id'], 'inner').groupBy('title').agg(avg('rating').alias('avg_rating')).orderBy(desc('avg_rating'), 'title').limit(1).select('title')

df_2 = users.join(movie_rating, users['user_id'] == movie_rating['user_id'], 'inner').groupBy('name').agg(count('name').alias('count')).orderBy(desc('count'), 'name').limit(1).select('name')

df_3 = df_2.union(df_1)

display(df_3)

name
Daniel
Joker


In [0]:
employee_data = [
    (5, "Max", "George", 26, "M", "Sales", "Sales", 1300, 200, 150, "Max@company.com", "California", "2638 Richards Avenue", 1),
    (13, "Katty", "Bond", 56, "F", "Manager", "Management", 150000, 0, 300, "Katty@company.com", "Arizona", "", 1),
    (11, "Richerd", "Gear", 57, "M", "Manager", "Management", 250000, 0, 300, "Richerd@company.com", "Alabama", "", 1),
    (10, "Jennifer", "Dion", 34, "F", "Sales", "Sales", 1000, 200, 150, "Jennifer@company.com", "Alabama", "", 13),
    (19, "George", "Joe", 50, "M", "Manager", "Management", 100000, 0, 300, "George@company.com", "Florida", "1003 Wyatt Street", 1),
    (18, "Laila", "Mark", 26, "F", "Sales", "Sales", 1000, 200, 150, "Laila@company.com", "Florida", "3655 Spirit Drive", 11),
    (20, "Sarrah", "Bicky", 31, "F", "Senior Sales", "Sales", 2000, 200, 150, "Sarrah@company.com", "Florida", "1176 Tyler Avenue", 19),
    (21, "Suzan", "Lee", 34, "F", "Sales", "Sales", 1300, 200, 150, "Suzan@company.com", "Florida", "1275 Monroe Avenue", 19),
    (22, "Mandy", "John", 31, "F", "Sales", "Sales", 1300, 200, 150, "Mandy@company.com", "Florida", "2510 Maryland Avenue", 19),
    (23, "Britney", "Berry", 45, "F", "Sales", "Sales", 1200, 200, 100, "Britney@company.com", "Florida", "3946 Steve Hunt Road", 19),
    (25, "Jack", "Mick", 29, "M", "Sales", "Sales", 1300, 200, 100, "Jack@company.com", "Hawaii", "3762 Stratford Drive", 19),
    (26, "Ben", "Ten", 43, "M", "Sales", "Sales", 1300, 150, 100, "Ben@company.com", "Hawaii", "3055 Indiana Avenue", 19),
    (27, "Tom", "Fridy", 32, "M", "Sales", "Sales", 1200, 200, 150, "Tom@company.com", "Hawaii", "801 Stratford Drive", 1),
    (29, "Antoney", "Adam", 34, "M", "Sales", "Sales", 1300, 180, 150, "Antoney@company.com", "Hawaii", "3533 Randall Drive", 1),
    (28, "Morgan", "Matt", 25, "M", "Sales", "Sales", 1200, 200, 150, "Morgan@company.com", "Hawaii", "2641 Randall Drive", 1),
    (6, "Molly", "Sam", 28, "F", "Sales", "Sales", 1400, 100, 150, "Molly@company.com", "Arizona", "3632 Polk Street", 13),
    (7, "Nicky", "Bat", 33, "F", "Sales", "Sales", 1400, 400, 100, "Molly@company.com", "Arizona", "3461 Preston Street", 13),
    (9, "Monika", "William", 33, "F", "Sales", "Sales", 1000, 200, 100, "Molly@company.com", "Alabama", "", 13),
    (17, "Mick", "Berry", 44, "M", "Senior Sales", "Sales", 2200, 200, 150, "Mick@company.com", "Florida", "", 11),
    (12, "Shandler", "Bing", 23, "M", "Auditor", "Audit", 1100, 200, 150, "Shandler@company.com", "Arizona", "", 11),
    (14, "Jason", "Tom", 23, "M", "Auditor", "Audit", 1000, 200, 150, "Jason@company.com", "Arizona", "", 11),
    (16, "Celine", "Anston", 27, "F", "Auditor", "Audit", 1000, 200, 150, "Celine@company.com", "Colorado", "", 11),
    (15, "Michale", "Jackson", 44, "F", "Auditor", "Audit", 700, 150, 150, "Michale@company.com", "Colorado", "", 11),
    (24, "Adam", "Morris", 30, "M", "Sales", "Sales", 1300, 200, 100, "Adam@company.com", "Alabama", "4541 Ferry Street", 19),
    (30, "Mark", "Jon", 28, "M", "Sales", "Sales", 1200, 200, 150, "Mark@company.com", "Alabama", "2522 George Avenue", 1),
    (8, "John", "Ford", 26, "M", "Senior Sales", "Sales", 1500, 140, 100, "Molly@company.com", "Alabama", "4832 New Creek Road", 13),
    (1, "Allen", "Wang", 55, "F", "Manager", "Management", 200000, 0, 300, "Allen@company.com", "California", "1069 Ventura Drive", 1),
    (2, "Joe", "Jack", 32, "M", "Sales", "Sales", 1000, 200, 150, "Joe@company.com", "California", "995 Jim Rosa Lane", 1),
    (3, "Henry", "Ted", 31, "M", "Senior Sales", "Sales", 2000, 200, 150, "Henry@company.com", "California", "1609 Ford Street", 1),
    (4, "Sam", "Mark", 25, "M", "Sales", "Sales", 1000, 120, 150, "Sam@company.com", "California", "4869 Libby Street", 1)
]

employee_schema = [
    "id",
    "first_name",
    "last_name",
    "age",
    "sex",
    "employee_title",
    "department",
    "salary",
    "target",
    "bonus",
    "email",
    "city",
    "address",
    "manager_id"
]

from pyspark.sql.functions import *
from pyspark.sql.types import *

tcs_employees = spark.createDataFrame(employee_data, employee_schema)
# display(tcs_employees)

managers = tcs_employees.alias('managers')
employees = tcs_employees.alias('employees')

employee_earning_manager_earning = employees.join(managers, col('employees.manager_id') == col('managers.id'), 'inner').select(col('employees.first_name').alias('EmployeeName'), col('employees.salary').alias('EmployeeSalary'), col('managers.first_name').alias('Manager'), col('managers.salary').alias('ManagerSalary'))

employee_earning_more_than_manager = employee_earning_manager_earning.filter(col('EmployeeSalary') > col('ManagerSalary'))

display(employee_earning_more_than_manager)

EmployeeName,EmployeeSalary,Manager,ManagerSalary
Richerd,250000,Allen,200000


In [0]:
employees1 = tcs_employees.alias('employees1')
employees2 = tcs_employees.alias('employees2')

employee_with_same_salary = employees1.join(employees2, (employees1['salary']==employees2['salary']) & (employees1['id'] != employees2['id']) & (employees1['first_name'] != employees2['first_name']), 'inner')\
    .select(employees1['id'], employees1['first_name'], employees1['salary'])\
    .dropDuplicates(['id'])\
    .orderBy(employees1['salary'].desc(), employees1['id'].asc())

display(employee_with_same_salary)

id,first_name,salary
3,Henry,2000
20,Sarrah,2000
6,Molly,1400
7,Nicky,1400
5,Max,1300
21,Suzan,1300
22,Mandy,1300
24,Adam,1300
25,Jack,1300
26,Ben,1300


In [0]:

second_highest_salary = employees.filter(col('salary') != employees.select(col('salary').alias('max_salary')).orderBy(col('max_salary').desc()).limit(1).collect()[0]['max_salary']).select(col('first_name'), col('salary')).orderBy(col('salary').desc()).limit(1)
display(second_highest_salary)

first_name,salary
Allen,200000


In [0]:
titanic_data = [
    (1, 0, 3, "Braund, Mr. Owen Harris", "male", 22, 1, 0, "A/5 21171", 7.25, None, "S"),
    (2, 1, 1, "Cumings, Mrs. John Bradley (Florence Briggs Thayer)", "female", 38, 1, 0, "PC 17599", 71.28, "C85", "C"),
    (3, 1, 3, "Heikkinen, Miss. Laina", "female", 26, 0, 0, "STON/O2. 3101282", 7.92, None, "S"),
    (4, 1, 1, "Futrelle, Mrs. Jacques Heath (Lily May Peel)", "female", 35, 1, 0, "113803", 53.1, "C123", "S"),
    (5, 0, 3, "Allen, Mr. William Henry", "male", 35, 0, 0, "373450", 8.05, None, "S"),
    (6, 0, 3, "Moran, Mr. James", "male", None, 0, 0, "330877", 8.46, None, "Q"),
    (7, 0, 1, "McCarthy, Mr. Timothy J", "male", 54, 0, 0, "17463", 51.86, "E46", "S"),
    (8, 0, 3, "Palsson, Master. Gosta Leonard", "male", 2, 3, 1, "349909", 21.07, None, "S"),
    (9, 1, 3, "Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)", "female", 27, 0, 2, "347742", 11.13, None, "S"),
    (10, 1, 2, "Nasser, Mrs. Nicholas (Adele Achem)", "female", 14, 1, 0, "237736", 30.07, None, "C"),
    (11, 1, 3, "Sandstrom, Miss. Marguerite Rut", "female", 4, 1, 1, "PP 9549", 16.7, "G6", "S"),
    (12, 1, 1, "Bonnell, Miss. Elizabeth", "female", 58, 0, 0, "113783", 26.55, "C103", "S"),
    (13, 0, 3, "Saundercock, Mr. William Henry", "male", 20, 0, 0, "A/5. 2151", 8.05, None, "S"),
    (14, 0, 3, "Andersson, Mr. Anders Johan", "male", 39, 1, 5, "347082", 31.27, None, "S"),
    (15, 0, 3, "Vestrom, Miss. Hulda Amanda Adolfina", "female", 14, 0, 0, "350406", 7.85, None, "S"),
    (16, 1, 2, "Hewlett, Mrs. (Mary D Kingcome)", "female", 55, 0, 0, "248706", 16.0, None, "S"),
    (17, 0, 3, "Rice, Master. Eugene", "male", 2, 4, 1, "382652", 29.12, None, "Q"),
    (18, 1, 2, "Williams, Mr. Charles Eugene", "male", None, 0, 0, "244373", 13.0, None, "S"),
    (19, 0, 3, "Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)", "female", 31, 1, 0, "345763", 18.0, None, "S"),
    (20, 1, 3, "Masselmani, Mrs. Fatima", "female", None, 0, 0, "2649", 7.22, None, "C"),
    (21, 0, 2, "Fynney, Mr. Joseph J", "male", 35, 0, 0, "239865", 26.0, None, "S"),
    (22, 1, 2, "Beesley, Mr. Lawrence", "male", 34, 0, 0, "248698", 13.0, "D56", "S"),
    (23, 1, 3, 'McGowan, Miss. Anna "Annie"', "female", 15, 0, 0, "330923", 8.03, None, "Q"),
    (24, 1, 1, "Sloper, Mr. William Thompson", "male", 28, 0, 0, "113788", 35.5, "A6", "S"),
    (25, 0, 3, "Palsson, Miss. Torborg Danira", "female", 8, 3, 1, "349909", 21.07, None, "S"),
    (26, 1, 3, "Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)", "female", 38, 1, 5, "347077", 31.39, None, "S"),
    (27, 0, 3, "Emir, Mr. Farred Chehab", "male", None, 0, 0, "2631", 7.22, None, "C"),
    (28, 0, 1, "Fortune, Mr. Charles Alexander", "male", 19, 3, 2, "19950", 263.0, "C23 C25 C27", "S"),
    (29, 1, 3, 'O\'Dwyer, Miss. Ellen "Nellie"', "female", None, 0, 0, "330959", 7.88, None, "Q"),
    (30, 0, 3, "Todoroff, Mr. Lalio", "male", None, 0, 0, "349216", 7.9, None, "S"),
    (31, 0, 1, "Uruchurtu, Don. Manuel E", "male", 40, 0, 0, "PC 17601", 27.72, None, "C"),
    (32, 1, 1, "Spencer, Mrs. William Augustus (Marie Eugenie)", "female", None, 1, 0, "PC 17569", 146.52, "B78", "C"),
    (33, 1, 3, "Glynn, Miss. Mary Agatha", "female", None, 0, 0, "335677", 7.75, None, "Q"),
    (34, 0, 2, "Wheadon, Mr. Edward H", "male", 66, 0, 0, "C.A. 24579", 10.5, None, "S"),
    (35, 0, 1, "Meyer, Mr. Edgar Joseph", "male", 28, 1, 0, "PC 17604", 82.17, None, "C"),
    (36, 0, 1, "Holverson, Mr. Alexander Oskar", "male", 42, 1, 0, "113789", 52.0, None, "S"),
    (37, 1, 3, "Mamee, Mr. Hanna", "male", None, 0, 0, "2677", 7.23, None, "C"),
    (38, 0, 3, "Cann, Mr. Ernest Charles", "male", 21, 0, 0, "A./5. 2152", 8.05, None, "S"),
    (39, 0, 3, "Vander Planke, Miss. Augusta Maria", "female", 18, 2, 0, "345764", 18.0, None, "S"),
    (40, 1, 3, "Nicola-Yarred, Miss. Jamila", "female", 14, 1, 0, "2651", 11.24, None, "C"),
    (41, 0, 3, "Ahlin, Mrs. Johan (Johanna Persdotter Larsson)", "female", 40, 1, 0, "7546", 9.47, None, "S"),
    (42, 0, 2, "Turpin, Mrs. William John Robert (Dorothy Ann Wonnacott)", "female", 27, 1, 0, "11668", 21.0, None, "S"),
    (43, 0, 3, "Kraeff, Mr. Theodor", "male", None, 0, 0, "349253", 7.9, None, "C"),
    (44, 1, 2, "Laroche, Miss. Simonne Marie Anne Andree", "female", 3, 1, 2, "SC/Paris 2123", 41.58, None, "C"),
    (45, 1, 3, "Devaney, Miss. Margaret Delia", "female", 19, 0, 0, "330958", 7.88, None, "Q"),
    (46, 0, 3, "Rogers, Mr. William John", "male", None, 0, 0, "S.C./A.4. 23567", 8.05, None, "S"),
    (47, 0, 3, "Lennon, Mr. Denis", "male", None, 1, 0, "370371", 15.5, None, "Q"),
    (48, 1, 3, "O'Driscoll, Miss. Bridget", "female", None, 0, 0, "14311", 7.75, None, "Q"),
    (49, 0, 3, "Samaan, Mr. Youssef", "male", None, 2, 0, "2662", 21.68, None, "C"),
    (50, 0, 3, "Arnold-Franchi, Mrs. Josef (Josefine Franchi)", "female", 18, 1, 0, "349237", 17.8, None, "S"),
    (51, 0, 3, "Panula, Master. Juha Niilo", "male", 7, 4, 1, "3101295", 39.69, None, "S"),
    (52, 0, 3, "Nosworthy, Mr. Richard Cater", "male", 21, 0, 0, "A/4. 39886", 7.8, None, "S"),
    (53, 1, 1, "Harper, Mrs. Henry Sleeper (Myna Haxtun)", "female", 49, 1, 0, "PC 17572", 76.73, "D33", "C"),
    (54, 1, 2, "Faunthorpe, Mrs. Lizzie (Elizabeth Anne Wilkinson)", "female", 29, 1, 0, "2926", 26.0, None, "S"),
    (55, 0, 1, "Ostby, Mr. Engelhart Cornelius", "male", 65, 0, 1, "113509", 61.98, "B30", "C"),
    (56, 1, 1, "Woolner, Mr. Hugh", "male", None, 0, 0, "19947", 35.5, "C52", "S"),
    (57, 1, 2, "Rugg, Miss. Emily", "female", 21, 0, 0, "C.A. 31026", 10.5, None, "S"),
    (58, 0, 3, "Novel, Mr. Mansouer", "male", 28.5, 0, 0, "2697", 7.23, None, "C"),
    (59, 1, 2, "West, Miss. Constance Mirium", "female", 5, 1, 2, "C.A. 34651", 27.75, None, "S"),
    (60, 0, 3, "Goodwin, Master. William Frederick", "male", 11, 5, 2, "CA 2144", 46.9, None, "S"),
    (61, 0, 3, "Sirayanian, Mr. Orsen", "male", 22, 0, 0, "2669", 7.23, None, "C"),
    (62, 1, 1, "Icard, Miss. Amelie", "female", 38, 0, 0, "113572", 80.0, "B28", None),
    (63, 0, 1, "Harris, Mr. Henry Birkhardt", "male", 45, 1, 0, "36973", 83.47, "C83", "S"),
    (64, 0, 3, "Skoog, Master. Harald", "male", 4, 3, 2, "347088", 27.9, None, "S"),
    (65, 0, 1, "Stewart, Mr. Albert A", "male", None, 0, 0, "PC 17605", 27.72, None, "C"),
    (66, 1, 3, "Moubarek, Master. Gerios", "male", None, 1, 1, "2661", 15.25, None, "C"),
    (67, 1, 2, "Nye, Mrs. (Elizabeth Ramell)", "female", 29, 0, 0, "C.A. 29395", 10.5, "F33", "S"),
    (68, 0, 3, "Crease, Mr. Ernest James", "male", 19, 0, 0, "S.P. 3464", 8.16, None, "S"),
    (69, 1, 3, "Andersson, Miss. Erna Alexandra", "female", 17, 4, 2, "3101281", 7.92, None, "S"),
    (70, 0, 3, "Kink, Mr. Vincenz", "male", 26, 2, 0, "315151", 8.66, None, "S"),
    (71, 0, 2, "Jenkin, Mr. Stephen Curnow", "male", 32, 0, 0, "C.A. 33111", 10.5, None, "S"),
    (72, 0, 3, "Goodwin, Miss. Lillian Amy", "female", 16, 5, 2, "CA 2144", 46.9, None, "S"),
    (73, 0, 2, "Hood, Mr. Ambrose Jr", "male", 21, 0, 0, "S.O.C. 14879", 73.5, None, "S"),
    (74, 0, 3, "Chronopoulos, Mr. Apostolos", "male", 26, 1, 0, "2680", 14.45, None, "C"),
    (75, 1, 3, "Bing, Mr. Lee", "male", 32, 0, 0, "1601", 56.5, None, "S"),
    (76, 0, 3, "Moen, Mr. Sigurd Hansen", "male", 25, 0, 0, "348123", 7.65, "F G73", "S"),
    (77, 0, 3, "Staneff, Mr. Ivan", "male", None, 0, 0, "349208", 7.9, None, "S"),
    (78, 0, 3, "Moutal, Mr. Rahamin Haim", "male", None, 0, 0, "374746", 8.05, None, "S"),
    (79, 1, 2, "Caldwell, Master. Alden Gates", "male", 0.83, 0, 2, "248738", 29.0, None, "S"),
    (80, 1, 3, "Dowdell, Miss. Elizabeth", "female", 30, 0, 0, "364516", 12.47, None, "S"),
    (81, 0, 3, "Waelens, Mr. Achille", "male", 22, 0, 0, "345767", 9.0, None, "S"),
    (82, 1, 3, "Sheerlinck, Mr. Jan Baptist", "male", 29, 0, 0, "345779", 9.5, None, "S"),
    (83, 1, 3, "McDermott, Miss. Brigdet Delia", "female", None, 0, 0, "330932", 7.79, None, "Q"),
    (84, 0, 1, "Carrau, Mr. Francisco M", "male", 28, 0, 0, "113059", 47.1, None, "S"),
    (85, 1, 2, "Ilett, Miss. Bertha", "female", 17, 0, 0, "SO/C 14885", 10.5, None, "S"),
    (86, 1, 3, "Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)", "female", 33, 3, 0, "3101278", 15.85, None, "S"),
    (87, 0, 3, "Ford, Mr. William Neal", "male", 16, 1, 3, "W./C. 6608", 34.38, None, "S"),
    (88, 0, 3, "Slocovski, Mr. Selman Francis", "male", None, 0, 0, "SOTON/OQ 392086", 8.05, None, "S"),
    (89, 1, 1, "Fortune, Miss. Mabel Helen", "female", 23, 3, 2, "19950", 263.0, "C23 C25 C27", "S"),
    (90, 0, 3, "Celotti, Mr. Francesco", "male", 24, 0, 0, "343275", 8.05, None, "S"),
    (91, 0, 3, "Christmann, Mr. Emil", "male", 29, 0, 0, "343276", 8.05, None, "S"),
    (92, 0, 3, "Andreasson, Mr. Paul Edvin", "male", 20, 0, 0, "347466", 7.85, None, "S"),
    (93, 0, 1, "Chaffee, Mr. Herbert Fuller", "male", 46, 1, 0, "W.E.P. 5734", 61.17, "E31", "S"),
    (94, 0, 3, "Dean, Mr. Bertram Frank", "male", 26, 1, 2, "C.A. 2315", 20.57, None, "S"),
    (95, 0, 3, "Coxon, Mr. Daniel", "male", 59, 0, 0, "364500", 7.25, None, "S"),
    (96, 0, 3, "Shorney, Mr. Charles Joseph", "male", None, 0, 0, "374910", 8.05, None, "S"),
    (97, 0, 1, "Goldschmidt, Mr. George B", "male", 71, 0, 0, "PC 17754", 34.65, "A5", "C"),
    (98, 1, 1, "Greenfield, Mr. William Bertram", "male", 23, 0, 1, "PC 17759", 63.36, "D10 D12", "C"),
    (99, 1, 2, "Doling, Mrs. John T (Ada Julia Bone)", "female", 34, 0, 1, "231919", 23.0, None, "S"),
    (100, 0, 2, "Kantor, Mr. Sinai", "male", 34, 1, 0, "244367", 26.0, None, "S")
]

titanic_schema = [
    "passengerid",
    "survived",
    "pclass",
    "name",
    "sex",
    "age",
    "sibsp",
    "parch",
    "ticket",
    "fare",
    "cabin",
    "embarked"
]

from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

titanic_struct_schema = StructType([
    StructField("passengerid", IntegerType(), True),
    StructField("survived", IntegerType(), True),
    StructField("pclass", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("sex", StringType(), True),
    StructField("age", DoubleType(), True),
    StructField("sibsp", IntegerType(), True),
    StructField("parch", IntegerType(), True),
    StructField("ticket", StringType(), True),
    StructField("fare", DoubleType(), True),
    StructField("cabin", StringType(), True),
    StructField("embarked", StringType(), True)
])

titanic = spark.createDataFrame(data=titanic_data, schema=titanic_struct_schema)
category_titanic_survived = titanic.withColumn('category', when(col('pclass') == 1, 'first_class').when(col('pclass') == 2, 'second_class').when(col('pclass') == 3, 'third_class').otherwise('Unknown'))

selectIve_from_category_titanic_survived = category_titanic_survived.select('category', 'survived')

selectIve_from_category_titanic_survived_1 = selectIve_from_category_titanic_survived.groupBy('survived').agg(
    sum(when(col('category') == 'first_class', 1)).alias('first_class'),
    sum(when(col('category') == 'second_class', 1)).alias('second_class'),
    sum(when(col('category') == 'third_class', 1)).alias('third_class')
)

display(selectIve_from_category_titanic_survived_1)

survived,first_class,second_class,third_class
0,11,6,42
1,10,12,19


In [0]:
schema = [
    "filename",
    "contents"
]

data = [
    (
        "draft1.txt",
        "The stock exchange predicts a bull market which would make many investors happy."
    ),
    (
        "draft2.txt",
        "The stock exchange predicts a bull market which would make many investors happy, but analysts warn of possibility of too much optimism and that in fact we are awaiting a bear market."
    ),
    (
        "final.txt",
        "The stock exchange predicts a bull market which would make many investors happy, but analysts warn of possibility of too much optimism and that in fact we are awaiting a bear market. As always predicting the future market is an uncertain game and all investors should follow their instincts and best practices."
    )
]

google_file_store = spark.createDataFrame(data=data, schema=schema)
google_file_store.createOrReplaceTempView('google_file_store')
# display(google_file_store)

result = spark.sql("""
                   WITH words_exploded AS (
                       SELECT LOWER(word) AS word
                       FROM google_file_store
                       LATERAL VIEW EXPLODE(SPLIT(REGEXP_REPLACE(contents, '[^a-zA-Z\\\\s]', ''), ' ')) AS word
                       WHERE LENGTH(word) > 0
                   )
                   SELECT word, COUNT(*) AS occurrences
                   FROM words_exploded
                   GROUP BY word
                   ORDER BY occurrences DESC
                   """)

from pyspark.sql.functions import *

result1 = google_file_store.select(explode(split(regexp_replace(col('contents'), '[^a-zA-Z\\s]', ''), ' ')).alias('word'))\
    .select(lower(col('word')).alias('word')).filter(length(col('word')) > 0)\
        .groupBy('word').agg(count('*').alias('occurrences'))\
            .orderBy(desc('occurrences'))

display(result1)

result2 = google_file_store.filter((lower(col('filename')).rlike('draft')) & (lower(col('contents')).rlike('optimism')))
display(result2)

word,occurrences
market,6
a,5
investors,4
the,4
of,4
and,4
exchange,3
bull,3
predicts,3
would,3


filename,contents
draft2.txt,"The stock exchange predicts a bull market which would make many investors happy, but analysts warn of possibility of too much optimism and that in fact we are awaiting a bear market."


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Read the parquet file
raw_df = spark.read.parquet("/Volumes/workspace/stratascratch/stratascratch/restaurant_inspections.parquet", header=True, inferSchema=True)

# Define proper column names
proper_columns = [
    "serial_number",
    "activity_date",
    "facility_name",
    "score",
    "grade",
    "service_code",
    "service_description",
    "employee_id",
    "facility_address",
    "facility_city",
    "facility_id",
    "facility_state",
    "facility_zip",
    "owner_id",
    "owner_name",
    "pe_description",
    "program_element_pe",
    "program_name",
    "program_status",
    "record_id"
]

# Rename all columns
los_angeles_restaurant_health_inspections = raw_df.toDF(*proper_columns)

# Filter using the properly named column
filtered_result = los_angeles_restaurant_health_inspections.filter(col('owner_name') == 'GLASSELL COFFEE SHOP LLC')
display(filtered_result)

serial_number,activity_date,facility_name,score,grade,service_code,service_description,employee_id,facility_address,facility_city,facility_id,facility_state,facility_zip,owner_id,owner_name,pe_description,program_element_pe,program_name,program_status,record_id
DAJ00E07B,2017-12-29,HABITAT COFFEE SHOP,95,A,1,ROUTINE INSPECTION,EE0000923,3708 N EAGLE ROCK BLVD,LOS ANGELES,FA0170465,CA,90065,OW0178123,GLASSELL COFFEE SHOP LLC,RESTAURANT (0-30) SEATS MODERATE RISK,1631,HABITAT COFFEE SHOP,ACTIVE,PR0160774
DA0UEVGFC,2017-05-09,HABITAT COFFEE SHOP,98,A,1,ROUTINE INSPECTION,EE0000923,3708 N EAGLE ROCK BLVD,LOS ANGELES,FA0170465,CA,90065,OW0178123,GLASSELL COFFEE SHOP LLC,RESTAURANT (0-30) SEATS MODERATE RISK,1631,HABITAT COFFEE SHOP,ACTIVE,PR0160774
DAPBHJLLF,2016-07-05,HABITAT COFFEE SHOP,95,A,1,ROUTINE INSPECTION,EE0000923,3708 N EAGLE ROCK BLVD,LOS ANGELES,FA0170465,CA,90065,OW0178123,GLASSELL COFFEE SHOP LLC,RESTAURANT (0-30) SEATS MODERATE RISK,1631,HABITAT COFFEE SHOP,ACTIVE,PR0160774
DACSKA8GC,2016-03-22,HABITAT COFFEE SHOP,95,A,1,ROUTINE INSPECTION,EE0000923,3708 N EAGLE ROCK BLVD,LOS ANGELES,FA0170465,CA,90065,OW0178123,GLASSELL COFFEE SHOP LLC,RESTAURANT (0-30) SEATS MODERATE RISK,1631,HABITAT COFFEE SHOP,ACTIVE,PR0160774
DAWPRXR8D,2016-01-11,HABITAT COFFEE SHOP,97,A,1,ROUTINE INSPECTION,EE0000923,3708 N EAGLE ROCK BLVD,LOS ANGELES,FA0170465,CA,90065,OW0178123,GLASSELL COFFEE SHOP LLC,RESTAURANT (0-30) SEATS MODERATE RISK,1631,HABITAT COFFEE SHOP,ACTIVE,PR0160774


In [0]:
filtere_result_1 = los_angeles_restaurant_health_inspections.filter((col('facility_name') == 'STREET CHURROS') & (col('score') < 95))\
    .select('activity_date', 'pe_description')
display(filtere_result_1)

activity_date,pe_description
2017-12-29,RESTAURANT (0-30) SEATS LOW RISK
2016-12-01,RESTAURANT (0-30) SEATS LOW RISK
2016-06-16,RESTAURANT (0-30) SEATS LOW RISK


In [0]:
sorted_out = los_angeles_restaurant_health_inspections.orderBy(col('score').desc())
# display(sorted_out)
filtered_result_2 = sorted_out.filter((col('score') == 100) & (col('facility_name').rlike('PANDA')) & (year(to_date(col('activity_date'), 'yyyy-MM-dd')) == 2016))
display(filtered_result_2)

serial_number,activity_date,facility_name,score,grade,service_code,service_description,employee_id,facility_address,facility_city,facility_id,facility_state,facility_zip,owner_id,owner_name,pe_description,program_element_pe,program_name,program_status,record_id
DABIFI10F,2016-12-14,PANDA CHINESE RESTAURANT,100,A,1,ROUTINE INSPECTION,EE0000798,4032 S WESTERN AVE,LOS ANGELES,FA0068234,CA,90062,OW0018189,K & D CHINESE FAST FOOD INC,RESTAURANT (0-30) SEATS HIGH RISK,1632,PANDA CHINESE RESTAURANT,ACTIVE,PR0017135
DASIBGFOE,2016-08-02,PANDA CHINESE RESTAURANT,100,A,1,ROUTINE INSPECTION,EE0000798,4032 S WESTERN AVE,LOS ANGELES,FA0068234,CA,90062,OW0018189,K & D CHINESE FAST FOOD INC,RESTAURANT (0-30) SEATS HIGH RISK,1632,PANDA CHINESE RESTAURANT,ACTIVE,PR0017135
DAXJ63WHI,2016-02-25,PANDA CHINESE RESTAURANT,100,A,1,ROUTINE INSPECTION,EE0000798,4032 S WESTERN AVE,LOS ANGELES,FA0068234,CA,90062,OW0018189,K & D CHINESE FAST FOOD INC,RESTAURANT (0-30) SEATS HIGH RISK,1632,PANDA CHINESE RESTAURANT,ACTIVE,PR0017135
